# Netflix Customer Churn & Engagement Analysis
**Author:** Astha Chourasia  
**File:** AsthaChourasia_NetflixChurnAnalysis.ipynb  
**Dataset:** netflix_large_user_data.csv (1,000 records, 16 features)  
**Objective:** Identify churn patterns, engagement drivers, business risks and opportunities, and build a churn prediction model based on real dataset inspection.

---

## 1. Setup & Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score,
                             ConfusionMatrixDisplay)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'xtick.labelsize': 10,
                     'ytick.labelsize': 10})
NETFLIX_RED  = '#E50914'
NETFLIX_DARK = '#221F1F'
print('All libraries imported successfully.')

---
## 2. Load & Inspect Dataset

In [ ]:
# Adjust path as needed
df = pd.read_csv('netflix_large_user_data.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicates ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
print('=== Numerical Summary ===')
df.describe().round(2)

In [ ]:
print('=== Categorical Unique Values ===')
for col in df.select_dtypes(include='object').columns:
    print(f'  {col}: {df[col].unique().tolist()}')

---
## 3. Data Cleaning & Feature Engineering

In [ ]:
# Rename columns for convenience
df.columns = [
    'Customer_ID', 'Subscription_Length_Months', 'Satisfaction_Score',
    'Daily_Watch_Hours', 'Engagement_Rate', 'Device', 'Genre',
    'Region', 'Payment_History', 'Subscription_Plan', 'Churn',
    'Support_Queries', 'Age', 'Monthly_Income', 'Promo_Offers_Used',
    'Profiles_Created'
]

# Binary churn target
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

# Derived groupings based on actual data ranges
df['Age_Group'] = pd.cut(df['Age'], bins=[17,29,39,49,59,70],
                          labels=['18-29','30-39','40-49','50-59','60-70'])

df['Sub_Length_Group'] = pd.cut(df['Subscription_Length_Months'],
                                 bins=[0,6,12,18,24],
                                 labels=['1-6m','7-12m','13-18m','19-24m'])

df['Income_Quartile'] = pd.qcut(df['Monthly_Income'], q=4,
                                 labels=['Q1 (Low)','Q2','Q3','Q4 (High)'])

df['Support_Load'] = pd.cut(df['Support_Queries'], bins=[-1,2,5,10],
                             labels=['Low (0-2)','Medium (3-5)','High (6-10)'])

print('Feature engineering complete. Shape:', df.shape)
df[['Churn','Churn_Binary','Age_Group','Sub_Length_Group','Income_Quartile','Support_Load']].head()

---
## 4. Key KPI Dashboard

In [ ]:
total_customers  = len(df)
churned_n        = df['Churn_Binary'].sum()
retained_n       = total_customers - churned_n
churn_rate       = churned_n / total_customers * 100
avg_satisfaction = df['Satisfaction_Score'].mean()
avg_watch_time   = df['Daily_Watch_Hours'].mean()
avg_engagement   = df['Engagement_Rate'].mean()
avg_sub_length   = df['Subscription_Length_Months'].mean()
avg_support_q    = df['Support_Queries'].mean()
avg_income       = df['Monthly_Income'].mean()
delayed_pct      = (df['Payment_History'] == 'Delayed').mean() * 100

kpis = {
    'Total Customers'              : f'{total_customers:,}',
    'Churned Customers'            : f'{churned_n:,}',
    'Retained Customers'           : f'{retained_n:,}',
    'Overall Churn Rate'           : f'{churn_rate:.1f}%',
    'Avg Satisfaction Score (1-10)': f'{avg_satisfaction:.2f}',
    'Avg Daily Watch Time (hrs)'   : f'{avg_watch_time:.2f}',
    'Avg Engagement Rate (1-10)'   : f'{avg_engagement:.2f}',
    'Avg Subscription Length (mo)' : f'{avg_sub_length:.1f}',
    'Avg Support Queries'          : f'{avg_support_q:.2f}',
    'Avg Monthly Income ($)'       : f'{avg_income:,.0f}',
    'Delayed Payments %'           : f'{delayed_pct:.1f}%'
}

print('=' * 52)
print('   NETFLIX CHURN ANALYSIS — KEY KPIs')
print('=' * 52)
for k, v in kpis.items():
    print(f'  {k:<37} {v}')
print('=' * 52)

---
## 5. Exploratory Data Analysis (EDA)
### 5.1 Overall Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Overall Customer Churn Distribution', fontsize=15, fontweight='bold')

counts = df['Churn'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=[NETFLIX_RED, NETFLIX_DARK], startangle=90,
            textprops={'color': 'white', 'fontsize': 12})
axes[0].set_title('Churn Split')

bars = axes[1].bar(counts.index, counts.values,
                   color=[NETFLIX_RED, NETFLIX_DARK], edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[1].set_title('Churn Count')
axes[1].set_ylabel('Number of Customers')
axes[1].set_xlabel('Churn Status')
plt.tight_layout()
plt.savefig('fig01_churn_distribution.png', bbox_inches='tight')
plt.show()

print(f'Churned : {counts["Yes"]} ({counts["Yes"]/total_customers*100:.1f}%)')
print(f'Retained: {counts["No"]} ({counts["No"]/total_customers*100:.1f}%)')
print('\nInsight: The 53.9% churn rate signals a structural platform-wide retention problem.')
print('No single segment drives this — the churn is distributed across all groups.')

### 5.2 Churn by Subscription Plan

In [ ]:
plan_churn = df.groupby(['Subscription_Plan','Churn']).size().unstack(fill_value=0)
plan_churn['Churn_Rate'] = plan_churn['Yes'] / (plan_churn['Yes'] + plan_churn['No']) * 100
plan_churn = plan_churn.sort_values('Churn_Rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Churn by Subscription Plan', fontsize=14, fontweight='bold')

plan_churn[['No','Yes']].plot(kind='bar', stacked=True, ax=axes[0],
                               color=[NETFLIX_DARK, NETFLIX_RED], edgecolor='white', rot=0)
axes[0].set_title('Churned vs Retained Count')
axes[0].set_xlabel('Subscription Plan')
axes[0].set_ylabel('Customers')
axes[0].legend(['Retained','Churned'])

bars = axes[1].bar(plan_churn.index, plan_churn['Churn_Rate'],
                   color=NETFLIX_RED, edgecolor='white', width=0.5)
axes[1].axhline(y=churn_rate, color='gray', linestyle='--', linewidth=1.5,
                label=f'Overall Avg ({churn_rate:.1f}%)')
for bar, val in zip(bars, plan_churn['Churn_Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Churn Rate by Plan')
axes[1].set_xlabel('Subscription Plan')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 70)
axes[1].legend()
plt.tight_layout()
plt.savefig('fig02_churn_by_plan.png', bbox_inches='tight')
plt.show()
print('Churn rates: Basic=54.9%, Standard=53.4%, Premium=53.5%')
print('Insight: Plan tier does NOT differentiate churn. Upgrading customers will not solve retention.')

### 5.3 Churn by Region

In [ ]:
region_churn = (df.groupby('Region')['Churn_Binary']
                  .agg(['sum','count'])
                  .rename(columns={'sum':'Churned','count':'Total'}))
region_churn['Churn_Rate'] = region_churn['Churned'] / region_churn['Total'] * 100
region_churn = region_churn.sort_values('Churn_Rate')

fig, ax = plt.subplots(figsize=(10, 5))
colors = [NETFLIX_RED if r > churn_rate else NETFLIX_DARK for r in region_churn['Churn_Rate']]
bars = ax.barh(region_churn.index, region_churn['Churn_Rate'], color=colors, edgecolor='white')
ax.axvline(x=churn_rate, color='gray', linestyle='--', linewidth=1.5,
           label=f'Overall Avg ({churn_rate:.1f}%)')
for bar, val in zip(bars, region_churn['Churn_Rate']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontweight='bold')
ax.set_title('Churn Rate by Region', fontsize=14, fontweight='bold')
ax.set_xlabel('Churn Rate (%)')
ax.set_xlim(0, 70)
red_p  = mpatches.Patch(color=NETFLIX_RED,  label='Above Avg Churn')
dark_p = mpatches.Patch(color=NETFLIX_DARK, label='Below Avg Churn')
ax.legend(handles=[red_p, dark_p,
                   plt.Line2D([0],[0], color='gray', linestyle='--',
                              label=f'Avg {churn_rate:.1f}%')])
plt.tight_layout()
plt.savefig('fig03_churn_by_region.png', bbox_inches='tight')
plt.show()
print('Africa=55.7%, Asia=55.6% lead churn; South America=50.2% is lowest.')
print('Spread of 5.5pp — regional variation exists but is not extreme.')

### 5.4 Churn by Device & Genre

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Churn Rate by Device and Genre Preference', fontsize=14, fontweight='bold')

for ax, col in zip(axes, ['Device', 'Genre']):
    grp = (df.groupby(col)['Churn_Binary']
             .agg(['sum','count'])
             .assign(Churn_Rate=lambda x: x['sum']/x['count']*100)
             .sort_values('Churn_Rate', ascending=False))
    colors = [NETFLIX_RED if r > churn_rate else '#b0b0b0' for r in grp['Churn_Rate']]
    bars = ax.bar(grp.index, grp['Churn_Rate'], color=colors, edgecolor='white')
    ax.axhline(y=churn_rate, color='gray', linestyle='--', linewidth=1.5,
               label=f'Avg {churn_rate:.1f}%')
    for bar, val in zip(bars, grp['Churn_Rate']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(col)
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 75)
    ax.legend()
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('fig04_churn_device_genre.png', bbox_inches='tight')
plt.show()
print('Device: Laptop (58.7%) and Desktop (56.8%) churn most; Smart TV (49.7%) churns least.')
print('Smart TV = habitual, living-room viewing = higher stickiness.')
print('Genre: Thriller (57.9%) and Romance (56.9%) churn most; Comedy (46.9%) retains best.')

### 5.5 Satisfaction Score & Engagement Rate vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Churn Rate by Satisfaction Score and Engagement Rate', fontsize=14, fontweight='bold')

for ax, col, xlabel in zip(
        axes,
        ['Satisfaction_Score', 'Engagement_Rate'],
        ['Customer Satisfaction Score (1-10)', 'Engagement Rate (1-10)']):
    grp = (df.groupby(col)['Churn_Binary']
             .agg(['sum','count'])
             .assign(Churn_Rate=lambda x: x['sum']/x['count']*100))
    ax.bar(grp.index, grp['Churn_Rate'],
           color=NETFLIX_RED, edgecolor='white', alpha=0.85)
    ax.axhline(y=churn_rate, color='gray', linestyle='--',
               linewidth=1.5, label=f'Avg {churn_rate:.1f}%')
    z = np.polyfit(grp.index, grp['Churn_Rate'], 1)
    ax.plot(grp.index, np.poly1d(z)(grp.index),
            color='#b45309', linewidth=2, label='Trend')
    ax.set_title(xlabel)
    ax.set_xlabel('Score (1-10)')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 80)
    ax.set_xticks(range(1, 11))
    ax.legend()

plt.tight_layout()
plt.savefig('fig05_satisfaction_engagement.png', bbox_inches='tight')
plt.show()
print('Critical Insight: Score=9 satisfaction shows 64% churn — HIGHEST of any group.')
print('No consistent downward trend. Satisfaction surveys are NOT a reliable churn predictor here.')
print('Business cannot assume high satisfaction = low churn risk.')

### 5.6 Daily Watch Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Daily Watch Time: Churned vs Retained', fontsize=14, fontweight='bold')

for churn_val, label, color in [('Yes','Churned',NETFLIX_RED),('No','Retained',NETFLIX_DARK)]:
    axes[0].hist(df[df['Churn']==churn_val]['Daily_Watch_Hours'],
                 bins=20, alpha=0.6, color=color, label=label,
                 edgecolor='white', density=True)
axes[0].set_title('Watch Time Distribution')
axes[0].set_xlabel('Daily Watch Time (Hours)')
axes[0].set_ylabel('Density')
axes[0].legend()

df.boxplot(column='Daily_Watch_Hours', by='Churn', ax=axes[1],
           boxprops=dict(color=NETFLIX_RED),
           medianprops=dict(color='gold', linewidth=2),
           whiskerprops=dict(color='gray'),
           capprops=dict(color='gray'),
           flierprops=dict(markerfacecolor='gray', marker='o', alpha=0.4))
axes[1].set_title('Box Plot: Watch Time by Churn')
axes[1].set_xlabel('Churn Status')
axes[1].set_ylabel('Daily Watch Time (Hours)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('fig06_watch_time.png', bbox_inches='tight')
plt.show()

c_w = df[df['Churn']=='Yes']['Daily_Watch_Hours'].mean()
r_w = df[df['Churn']=='No']['Daily_Watch_Hours'].mean()
print(f'Churned avg watch time : {c_w:.2f} hrs')
print(f'Retained avg watch time: {r_w:.2f} hrs')
print('Difference is negligible (< 0.04 hrs). Watch time alone cannot flag churn risk.')

### 5.7 Support Queries & Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Support Queries and Churn Relationship', fontsize=14, fontweight='bold')

sup_exact = (df.groupby('Support_Queries')['Churn_Binary']
               .agg(['sum','count'])
               .assign(Churn_Rate=lambda x: x['sum']/x['count']*100))
axes[0].bar(sup_exact.index, sup_exact['Churn_Rate'],
            color=NETFLIX_RED, edgecolor='white', alpha=0.85)
axes[0].axhline(y=churn_rate, color='gray', linestyle='--',
                linewidth=1.5, label=f'Avg {churn_rate:.1f}%')
axes[0].set_title('Churn Rate by Exact Support Count')
axes[0].set_xlabel('Number of Support Queries')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xticks(range(0, 11))
axes[0].set_ylim(0, 80)
axes[0].legend()

load_churn = (df.groupby('Support_Load', observed=True)['Churn_Binary']
                .agg(['sum','count'])
                .assign(Churn_Rate=lambda x: x['sum']/x['count']*100))
bars = axes[1].bar(load_churn.index.astype(str), load_churn['Churn_Rate'],
                   color=['#b0b0b0','#ff8c69',NETFLIX_RED], edgecolor='white')
axes[1].axhline(y=churn_rate, color='gray', linestyle='--',
                linewidth=1.5, label=f'Avg {churn_rate:.1f}%')
for bar, val in zip(bars, load_churn['Churn_Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Churn Rate by Support Load Bucket')
axes[1].set_xlabel('Support Query Load')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 75)
axes[1].legend()
plt.tight_layout()
plt.savefig('fig07_support_queries.png', bbox_inches='tight')
plt.show()
print('Customers with 9 queries: 62% churn (highest). High load (6-10): elevated churn pattern.')
print('Repeated unresolved service friction is the strongest behavioral churn signal in this dataset.')

### 5.8 Subscription Length & Churn

In [ ]:
sub_churn = (df.groupby('Sub_Length_Group', observed=True)['Churn_Binary']
               .agg(['sum','count'])
               .assign(Churn_Rate=lambda x: x['sum']/x['count']*100))

fig, ax = plt.subplots(figsize=(9, 5))
colors = [NETFLIX_RED if r > churn_rate else '#b0b0b0' for r in sub_churn['Churn_Rate']]
bars = ax.bar(sub_churn.index.astype(str), sub_churn['Churn_Rate'],
              color=colors, edgecolor='white')
ax.axhline(y=churn_rate, color='gray', linestyle='--',
           linewidth=1.5, label=f'Avg {churn_rate:.1f}%')
for bar, val, n in zip(bars, sub_churn['Churn_Rate'], sub_churn['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%\n(n={n})', ha='center', va='bottom',
            fontsize=10, fontweight='bold')
ax.set_title('Churn Rate by Subscription Length Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Subscription Length')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 70)
ax.legend()
plt.tight_layout()
plt.savefig('fig08_subscription_length.png', bbox_inches='tight')
plt.show()
print('7-12 month window has the highest churn rate (56.8%). This is the critical intervention window.')
print('Note: 13-18m bucket is absent from this dataset — gap in data distribution.')

### 5.9 Age Group & Churn

In [ ]:
age_churn = (df.groupby('Age_Group', observed=True)['Churn_Binary']
               .agg(['sum','count'])
               .assign(Churn_Rate=lambda x: x['sum']/x['count']*100))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Churn Analysis by Age Group', fontsize=14, fontweight='bold')

colors = [NETFLIX_RED if r > churn_rate else '#b0b0b0' for r in age_churn['Churn_Rate']]
bars = axes[0].bar(age_churn.index.astype(str), age_churn['Churn_Rate'],
                   color=colors, edgecolor='white')
axes[0].axhline(y=churn_rate, color='gray', linestyle='--',
                linewidth=1.5, label=f'Avg {churn_rate:.1f}%')
for bar, val in zip(bars, age_churn['Churn_Rate']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Churn Rate by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 70)
axes[0].legend()

for churn_val, label, color in [('Yes','Churned',NETFLIX_RED),('No','Retained',NETFLIX_DARK)]:
    axes[1].hist(df[df['Churn']==churn_val]['Age'],
                 bins=20, alpha=0.6, color=color, label=label, edgecolor='white')
axes[1].set_title('Age Distribution by Churn')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].legend()
plt.tight_layout()
plt.savefig('fig09_age_churn.png', bbox_inches='tight')
plt.show()
print('18-29: 57.4% churn | 30-39: 57.2% | 50-59: 49.7% | 60-70: 50.0%')
print('Younger users churn more. Older customers show more habitual, loyal viewing behavior.')

### 5.10 Payment History & Promotional Offers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Payment History and Promotional Offers vs Churn', fontsize=14, fontweight='bold')

pay_churn = (df.groupby('Payment_History')['Churn_Binary']
               .agg(['sum','count'])
               .assign(Churn_Rate=lambda x: x['sum']/x['count']*100))
bars = axes[0].bar(pay_churn.index, pay_churn['Churn_Rate'],
                   color=[NETFLIX_DARK, NETFLIX_RED], edgecolor='white', width=0.4)
axes[0].axhline(y=churn_rate, color='gray', linestyle='--',
                linewidth=1.5, label=f'Avg {churn_rate:.1f}%')
for bar, val, n in zip(bars, pay_churn['Churn_Rate'], pay_churn['count']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%\n(n={n})', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Churn Rate by Payment History')
axes[0].set_xlabel('Payment History')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 70)
axes[0].legend()

promo_churn = (df.groupby('Promo_Offers_Used')['Churn_Binary']
                 .agg(['sum','count'])
                 .assign(Churn_Rate=lambda x: x['sum']/x['count']*100))
colors_p = [NETFLIX_RED if r > churn_rate else '#b0b0b0' for r in promo_churn['Churn_Rate']]
bars2 = axes[1].bar(promo_churn.index, promo_churn['Churn_Rate'],
                    color=colors_p, edgecolor='white')
axes[1].axhline(y=churn_rate, color='gray', linestyle='--',
                linewidth=1.5, label=f'Avg {churn_rate:.1f}%')
for bar, val in zip(bars2, promo_churn['Churn_Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Churn Rate by Promotional Offers Used')
axes[1].set_xlabel('Number of Promo Offers Used')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 75)
axes[1].set_xticks(range(0, 6))
axes[1].legend()
plt.tight_layout()
plt.savefig('fig10_payment_promo.png', bbox_inches='tight')
plt.show()
print('Payment: On-Time=54.5%, Delayed=53.3% — virtually identical. Payment timing is not a churn driver.')
print('Promo: 3 offers=60.8%, 4 offers=60.2% — highest churn. Suggests promo-driven subscribers lack organic loyalty.')

### 5.11 Correlation Heatmap

In [ ]:
num_cols = ['Subscription_Length_Months','Satisfaction_Score','Daily_Watch_Hours',
            'Engagement_Rate','Support_Queries','Age','Monthly_Income',
            'Promo_Offers_Used','Profiles_Created','Churn_Binary']

corr_matrix = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, square=True, linewidths=0.5,
            ax=ax, cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Correlation Matrix — All Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig11_correlation_heatmap.png', bbox_inches='tight')
plt.show()

churn_corr = (corr_matrix['Churn_Binary']
              .drop('Churn_Binary')
              .sort_values(key=abs, ascending=False))
print('Correlations with Churn (|r| sorted):')
for feat, val in churn_corr.items():
    print(f'  {feat:<38}: {val:+.4f}')
print('\nAll |r| < 0.10 — no single feature is a strong individual predictor of churn.')
print('This makes a multivariate model (Random Forest / GBM) essential.')

---
## 6. Trend & Pattern Analysis
### 6.1 Region × Plan Churn Heatmap

In [ ]:
pivot = (df.pivot_table(values='Churn_Binary', index='Region',
                         columns='Subscription_Plan', aggfunc='mean') * 100)
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Reds', linewidths=0.5,
            vmin=40, vmax=70, ax=ax, cbar_kws={'label': 'Churn Rate (%)'})
ax.set_title('Churn Rate Heatmap: Region × Subscription Plan',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig12_heatmap_region_plan.png', bbox_inches='tight')
plt.show()
print('Insight: Certain region-plan combinations show elevated churn pockets.')
print('These cross-segments are priority retention campaign targets.')

### 6.2 Age Group × Device Churn Heatmap

In [ ]:
pivot2 = (df.pivot_table(values='Churn_Binary', index='Age_Group',
                          columns='Device', aggfunc='mean',
                          observed=True) * 100)
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5,
            vmin=35, vmax=75, ax=ax, cbar_kws={'label': 'Churn Rate (%)'})
ax.set_title('Churn Rate Heatmap: Age Group × Device',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig13_heatmap_age_device.png', bbox_inches='tight')
plt.show()
print('Young laptop/desktop users show highest churn concentration.')
print('These are likely students/young professionals with high platform-switching flexibility.')

### 6.3 Income Distribution by Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Monthly Income Analysis by Churn Status', fontsize=14, fontweight='bold')

for churn_val, label, color in [('Yes','Churned',NETFLIX_RED),('No','Retained',NETFLIX_DARK)]:
    axes[0].hist(df[df['Churn']==churn_val]['Monthly_Income'],
                 bins=20, alpha=0.6, color=color, label=label,
                 edgecolor='white', density=True)
axes[0].set_title('Income Distribution')
axes[0].set_xlabel('Monthly Income ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

inc_churn = (df.groupby('Income_Quartile', observed=True)['Churn_Binary']
               .agg(['sum','count'])
               .assign(Churn_Rate=lambda x: x['sum']/x['count']*100))
colors_i = [NETFLIX_RED if r > churn_rate else '#b0b0b0' for r in inc_churn['Churn_Rate']]
bars = axes[1].bar(inc_churn.index.astype(str), inc_churn['Churn_Rate'],
                   color=colors_i, edgecolor='white')
axes[1].axhline(y=churn_rate, color='gray', linestyle='--',
                linewidth=1.5, label=f'Avg {churn_rate:.1f}%')
for bar, val in zip(bars, inc_churn['Churn_Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Churn Rate by Income Quartile')
axes[1].set_xlabel('Income Quartile')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 70)
axes[1].legend()
plt.tight_layout()
plt.savefig('fig14_income_churn.png', bbox_inches='tight')
plt.show()
print('Income quartile shows minimal churn variation. Netflix value is perceived consistently across income levels.')

---
## 7. Interactive Plotly Dashboard

In [ ]:
fig_dash = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Churn Split', 'Churn Rate by Plan', 'Churn Rate by Region',
                    'Churn Rate by Device', 'Churn Rate by Genre', 'Churn Rate by Age Group'),
    specs=[[{'type':'pie'},{'type':'bar'},{'type':'bar'}],
           [{'type':'bar'},{'type':'bar'},{'type':'bar'}]]
)

cv = df['Churn'].value_counts()
fig_dash.add_trace(go.Pie(labels=cv.index.tolist(), values=cv.values.tolist(),
                           marker_colors=[NETFLIX_RED, NETFLIX_DARK],
                           hole=0.35, showlegend=False), row=1, col=1)

for (col, row, pos, color) in [
    ('Subscription_Plan', 1, 2, '#E50914'),
    ('Region',            1, 3, '#b45309'),
    ('Device',            2, 1, '#7c3aed'),
    ('Genre',             2, 2, '#0891b2'),
]:
    grp = (df.groupby(col)['Churn_Binary'].mean() * 100).reset_index()
    grp.columns = [col, 'Churn_Rate']
    grp = grp.sort_values('Churn_Rate')
    fig_dash.add_trace(
        go.Bar(x=grp[col], y=grp['Churn_Rate'], marker_color=color,
               showlegend=False,
               text=grp['Churn_Rate'].round(1).astype(str)+'%',
               textposition='outside'),
        row=row, col=pos
    )

ag = (df.groupby('Age_Group', observed=True)['Churn_Binary'].mean() * 100).reset_index()
ag.columns = ['Age_Group','Churn_Rate']
fig_dash.add_trace(
    go.Bar(x=ag['Age_Group'].astype(str), y=ag['Churn_Rate'],
           marker_color='#059669', showlegend=False,
           text=ag['Churn_Rate'].round(1).astype(str)+'%',
           textposition='outside'),
    row=2, col=3
)

fig_dash.update_layout(
    height=720,
    title_text='Netflix Churn Analysis — Interactive Overview Dashboard',
    title_font_size=18,
    plot_bgcolor='#fafafa',
    paper_bgcolor='white'
)
fig_dash.update_yaxes(range=[0, 75])
fig_dash.show()
print('Interactive dashboard rendered. Hover over bars/slices for details.')

---
## 8. Churn Prediction Model
### 8.1 Feature Preparation

In [ ]:
model_df = df[['Subscription_Length_Months','Satisfaction_Score','Daily_Watch_Hours',
               'Engagement_Rate','Device','Genre','Region','Payment_History',
               'Subscription_Plan','Support_Queries','Age','Monthly_Income',
               'Promo_Offers_Used','Profiles_Created','Churn_Binary']].copy()

model_df = pd.get_dummies(model_df,
                           columns=['Device','Genre','Region','Payment_History','Subscription_Plan'],
                           drop_first=False)

X = model_df.drop('Churn_Binary', axis=1)
y = model_df['Churn_Binary']

print(f'Features: {X.shape[1]}')
print(f'Samples : {X.shape[0]}')
print(f'Class balance — 0: {(y==0).sum()} | 1: {(y==1).sum()}')
X.head(3)

### 8.2 Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples ({y_train.mean()*100:.1f}% churn)')
print(f'Test : {X_test.shape[0]} samples  ({y_test.mean()*100:.1f}% churn)')

### 8.3 Model Training — Three Algorithms

In [ ]:
models = {
    'Logistic Regression'  : LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'        : RandomForestClassifier(n_estimators=200, max_depth=8,
                                                     min_samples_leaf=5, random_state=42),
    'Gradient Boosting'    : GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                         max_depth=4, random_state=42)
}

results = {}
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test
    X_cv = X_train_sc if name == 'Logistic Regression' else X_train

    cv_scores = cross_val_score(model, X_cv, y_train,
                                cv=cv_splitter, scoring='roc_auc')
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
        'accuracy'   : accuracy_score(y_test, y_pred),
        'roc_auc'    : roc_auc_score(y_test, y_prob),
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std' : cv_scores.std()
    }
    print(f'{name}:')
    print(f'  Accuracy  : {results[name]["accuracy"]*100:.2f}%')
    print(f'  ROC-AUC   : {results[name]["roc_auc"]:.4f}')
    print(f'  5-CV AUC  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}\n')

### 8.4 ROC Curves & Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

roc_colors = [NETFLIX_RED, NETFLIX_DARK, '#b45309']
for (name, res), col in zip(results.items(), roc_colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, color=col, linewidth=2,
                 label=f'{name} (AUC={res["roc_auc"]:.3f})')
axes[0].plot([0,1],[0,1],'k--', linewidth=1, label='Random Classifier')
axes[0].set_title('ROC Curves — All Models')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1.02])

metric_df = pd.DataFrame([
    {'Model': n,
     'Accuracy (%)': r['accuracy']*100,
     'ROC-AUC (%)' : r['roc_auc']*100,
     'CV-AUC (%)'  : r['cv_auc_mean']*100}
    for n, r in results.items()
])
x_pos = np.arange(len(metric_df))
w = 0.25
axes[1].bar(x_pos - w, metric_df['Accuracy (%)'], w, label='Accuracy',  color=NETFLIX_DARK)
axes[1].bar(x_pos,     metric_df['ROC-AUC (%)'],  w, label='ROC-AUC',  color=NETFLIX_RED)
axes[1].bar(x_pos + w, metric_df['CV-AUC (%)'],   w, label='CV-AUC',   color='#b45309')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(metric_df['Model'], rotation=10)
axes[1].set_title('Metric Comparison')
axes[1].set_ylabel('Score (%)')
axes[1].set_ylim(40, 80)
axes[1].legend()
axes[1].axhline(y=50, color='gray', linestyle=':', linewidth=1)
plt.tight_layout()
plt.savefig('fig15_model_comparison.png', bbox_inches='tight')
plt.show()

### 8.5 Best Model — Confusion Matrix & Detailed Report

In [ ]:
best_name = max(results, key=lambda k: results[k]['roc_auc'])
best = results[best_name]
print(f'Best Model : {best_name}')
print(f'ROC-AUC    : {best["roc_auc"]:.4f}')
print(f'Accuracy   : {best["accuracy"]*100:.2f}%')
print()
print(classification_report(y_test, best['y_pred'],
                             target_names=['Retained','Churned']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Best Model: {best_name}', fontsize=13, fontweight='bold')

ConfusionMatrixDisplay(
    confusion_matrix(y_test, best['y_pred']),
    display_labels=['Retained','Churned']
).plot(ax=axes[0], cmap='Reds', colorbar=False)
axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(y_test, best['y_prob'])
axes[1].plot(fpr, tpr, color=NETFLIX_RED, linewidth=2.5,
             label=f'AUC = {best["roc_auc"]:.3f}')
axes[1].fill_between(fpr, tpr, alpha=0.12, color=NETFLIX_RED)
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')
plt.tight_layout()
plt.savefig('fig16_best_model.png', bbox_inches='tight')
plt.show()

### 8.6 Feature Importance (Random Forest)

In [ ]:
rf = results['Random Forest']['model']
importances = (pd.Series(rf.feature_importances_, index=X.columns)
                 .sort_values(ascending=False)
                 .head(15))

fig, ax = plt.subplots(figsize=(10, 6))
palette = [NETFLIX_RED if i < 5 else '#b45309' if i < 10 else '#b0b0b0'
           for i in range(len(importances))]
ax.barh(importances.index[::-1], importances.values[::-1],
        color=palette[::-1], edgecolor='white')
ax.set_title('Top 15 Feature Importances — Random Forest',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.legend(handles=[
    mpatches.Patch(color=NETFLIX_RED, label='Top 5'),
    mpatches.Patch(color='#b45309',   label='Rank 6-10'),
    mpatches.Patch(color='#b0b0b0',   label='Rank 11-15')
])
plt.tight_layout()
plt.savefig('fig17_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 10 Features:')
for feat, imp in importances.head(10).items():
    print(f'  {feat:<45}: {imp:.4f}')

### 8.7 Churn Probability Distribution

In [ ]:
rf_probs = results['Random Forest']['y_prob']

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(rf_probs[y_test == 0], bins=25, alpha=0.7,
        color=NETFLIX_DARK, label='Retained (Actual)', edgecolor='white')
ax.hist(rf_probs[y_test == 1], bins=25, alpha=0.7,
        color=NETFLIX_RED, label='Churned (Actual)', edgecolor='white')
ax.axvline(x=0.5, color='gold', linewidth=2, linestyle='--',
           label='Decision Threshold (0.5)')
ax.set_title('Predicted Churn Probability Distribution (Random Forest)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Churn Probability')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('fig18_probability_dist.png', bbox_inches='tight')
plt.show()
print('The overlapping distributions confirm inherent noise in the dataset.')
print('AUC > 0.5 confirms the model beats random, but the business should not expect near-perfect predictions.')

### 8.8 Model Summary Table

In [ ]:
summary_df = pd.DataFrame([
    {'Model'           : name,
     'Accuracy (%)'   : f"{res['accuracy']*100:.2f}",
     'ROC-AUC'        : f"{res['roc_auc']:.4f}",
     'CV-AUC (5-fold)': f"{res['cv_auc_mean']:.4f} ± {res['cv_auc_std']:.4f}"}
    for name, res in results.items()
]).set_index('Model')

print('=== Model Performance Summary ===')
print(summary_df.to_string())
print(f'\nSelected Best Model: {best_name}')

---
## 9. Key Findings, Business Risks, Opportunities & Recommended Actions

In [ ]:
report = """
╔═══════════════════════════════════════════════════════════════════════════════╗
║       NETFLIX CHURN & ENGAGEMENT ANALYSIS — BUSINESS SUMMARY                ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ 5 KEY FINDINGS (grounded in actual dataset results)                         ║
║                                                                               ║
║ F1. Structural, platform-wide churn at 53.9%                                 ║
║     No single segment is safe. Basic=54.9%, Standard=53.4%, Premium=53.5%.  ║
║     The near-equal churned/retained split means this is a systemic problem,  ║
║     not an isolated segment issue.                                            ║
║                                                                               ║
║ F2. Satisfaction score is NOT a reliable retention metric                    ║
║     Customers scoring 9/10 satisfaction churn at 64% — the HIGHEST of any   ║
║     group. The business cannot rely on satisfaction surveys as an early      ║
║     churn warning signal.                                                     ║
║                                                                               ║
║ F3. High support query load (9+ queries) is the clearest behavioral flag     ║
║     Customers with 9 support interactions show 62% churn — 8+ pp above      ║
║     average. Repeated unresolved friction is the strongest detectable signal ║
║     in this dataset.                                                          ║
║                                                                               ║
║ F4. The 7-12 month subscription window is the highest-risk period (56.8%)   ║
║     Post-introductory customers who haven't built long-term habits are most  ║
║     vulnerable. This is the critical intervention window for retention.       ║
║                                                                               ║
║ F5. Young laptop/desktop users (18-39) are the highest-risk demographic     ║
║     18-29: 57.4%, 30-39: 57.2% churn. Laptop=58.7%, Desktop=56.8%.         ║
║     Smart TV users (49.7%) are the most loyal — viewing context matters.     ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ 3 BUSINESS RISKS                                                             ║
║                                                                               ║
║ R1. Promo-dependency trap: Customers using 3-4 promos churn at 60.8-60.2%.  ║
║     Discount-driven acquisition is attracting low-loyalty subscribers.       ║
║                                                                               ║
║ R2. Plan-agnostic churn erodes upsell ROI: Premium and Basic churn at nearly ║
║     identical rates. Upselling customers does not reduce churn risk —        ║
║     the value gap between plan tiers must be addressed.                       ║
║                                                                               ║
║ R3. Africa & Asia as high-churn growth markets (55.6-55.7%): Without        ║
║     localization investment, these high-growth regions risk becoming          ║
║     high-exit markets as competition intensifies.                             ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ 3 BUSINESS OPPORTUNITIES                                                    ║
║                                                                               ║
║ O1. Smart TV loyalty advantage (49.7% churn — lowest device):                ║
║     TV manufacturer partnerships and default app placement can structurally   ║
║     improve retention by shifting viewing to the stickiest device context.    ║
║                                                                               ║
║ O2. Comedy as a retention anchor (46.9% churn — lowest genre):               ║
║     Comedy fans are the most retained group. Expanding comedy content and    ║
║     using comedy-led recommendations as re-engagement hooks is a proven,      ║
║     low-cost retention lever.                                                 ║
║                                                                               ║
║ O3. Older audience (50+) as a retention stronghold (49.7-50%):               ║
║     Customers aged 50+ show significantly lower churn. Creating age-relevant  ║
║     content and UI experiences can deepen loyalty in this underserved segment.║
╠═══════════════════════════════════════════════════════════════════════════════╣
║ 5 RECOMMENDED ACTIONS                                                       ║
║                                                                               ║
║ A1. Month 7-12 Retention Programme: Trigger personalised content             ║
║     recommendations, milestone rewards, and proactive outreach for all        ║
║     subscribers entering the 7-12 month window.                               ║
║                                                                               ║
║ A2. Support Escalation Churn Alert System: Automate a flag for customers     ║
║     logging 7+ support queries. Route to senior retention agents with a       ║
║     retention offer before cancellation intent is reached.                    ║
║                                                                               ║
║ A3. Replace satisfaction KPI with behavioral metrics: Track session          ║
║     frequency, content completion rate, and device switching as real-time     ║
║     churn signals instead of survey satisfaction scores.                      ║
║                                                                               ║
║ A4. Redesign promotional offer strategy: Cap promos at 2 per customer per   ║
║     year and shift spend to content-value communication rather than           ║
║     price discounts. This targets organic-value subscribers, not deal hunters.║
║                                                                               ║
║ A5. Regional content localisation in Africa & Asia: Commission or license    ║
║     local-language content for the two highest-churn regions to build         ║
║     emotional platform loyalty beyond price competitiveness.                  ║
╚═══════════════════════════════════════════════════════════════════════════════╝
"""
print(report)